# Custom Model B: Dense Embeddings (Sentence-Transformers)

In this notebook, we:
1. Reload the preprocessed reviews and setup the same Train/Test split.
2. Vectorize the text using an advanced pre-trained NLP Transformer model (`all-MiniLM-L6-v2` via Hugging Face `sentence-transformers`).
   *Unlike TF-IDF, these embeddings map semantic intent. Words that occur in the same context (Distributional Hypothesis) map closely in Vector Space!*
3. Compute Cosine Similarities between queries and test locations.
4. Evaluate the ranked results using the exact same metrics.

In [6]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import time
import warnings
warnings.filterwarnings('ignore')

### 1. Load Data & Train/Test Split (Same Seed as Baseline)

In [7]:
# Load prepared data
df_reviews = pd.read_csv('prepared_reviews.csv')
df_places = pd.read_csv('../data/Tripadvisor.csv', low_memory=False)

eval_cols = ['id', 'typeR', 'activiteSubType', 'restaurantType', 'restaurantTypeCuisine', 'priceRange']
df_places = df_places[eval_cols].copy()

df_merged = pd.merge(df_reviews, df_places, left_on='idplace', right_on='id', how='inner')

# Split
np.random.seed(42)
shuffled_indices = np.random.permutation(len(df_merged))
split_idx = len(df_merged) // 2

train_df = df_merged.iloc[shuffled_indices[:split_idx]].copy()
test_df = df_merged.iloc[shuffled_indices[split_idx:]].copy().reset_index(drop=True)

### 2. Evaluation Functions

In [8]:
def eval_level_1(query_typeR, sorted_test_typeR_list):
    if pd.isna(query_typeR) or query_typeR not in sorted_test_typeR_list:
        return None
    for i, t in enumerate(sorted_test_typeR_list):
        if t == query_typeR:
            return i
    return None

def extract_subcategories(row):
    cats = []
    cols = ['activiteSubType', 'restaurantType', 'restaurantTypeCuisine', 'priceRange']
    for c in cols:
        val = row[c]
        if pd.notna(val):
            parts = str(val).split(',')
            for p in parts:
                cats.append(p.strip().lower())
    return set(cats)

test_subcats_list = [extract_subcategories(row) for _, row in test_df.iterrows()]

def eval_level_2(query_subcats, sorted_test_indices):
    if not query_subcats:
        return None
    for i, test_idx in enumerate(sorted_test_indices):
        test_sub = test_subcats_list[test_idx]
        if len(query_subcats.intersection(test_sub)) > 0:
            return i
    return None

### 3. Model Pipeline: Sentence Transformers (Hugging Face)

In [9]:
model = SentenceTransformer('all-MiniLM-L6-v2')

# Use keyword signatures for fair comparison with BM25/TF-IDF
test_texts = test_df['top_100_words'].fillna('').tolist()
train_texts = train_df['top_100_words'].fillna('').tolist()

print("Encoding 50% Test set...")
test_vectors = model.encode(test_texts, show_progress_bar=True)
print("Encoding 50% Train set (Queries)...")
train_vectors = model.encode(train_texts, show_progress_bar=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding 50% Test set...


Batches:   0%|          | 0/29 [00:00<?, ?it/s]

Encoding 50% Train set (Queries)...


Batches:   0%|          | 0/29 [00:00<?, ?it/s]

### 4. Running the Benchmark

In [10]:
lvl1_errors = []
lvl2_errors = []

start_time = time.time()
print(f"Evaluating {len(train_df)} queries...")

similarities = cosine_similarity(train_vectors, test_vectors)
test_type_array = test_df['typeR'].values

for i in range(len(train_df)):
    row = train_df.iloc[i]
    sims = similarities[i]
    
    # Sort descending
    ranked_indices = np.argsort(sims)[::-1]
    test_typeR_list = test_type_array[ranked_indices].tolist()
    
    err_1 = eval_level_1(row['typeR'], test_typeR_list)
    if err_1 is not None:
        lvl1_errors.append(err_1)
        
    query_subcats = extract_subcategories(row)
    err_2 = eval_level_2(query_subcats, ranked_indices)
    if err_2 is not None:
        lvl2_errors.append(err_2)

print(f"Evaluation done in {time.time() - start_time:.2f}s")
print("-" * 30)
print(f"Model B (HuggingFace Transformers) Average Ranking Error Level 1: {np.mean(lvl1_errors):.2f}")
print(f"Model B (HuggingFace Transformers) Average Ranking Error Level 2: {np.mean(lvl2_errors):.2f}")

Evaluating 917 queries...
Evaluation done in 0.18s
------------------------------
Model B (HuggingFace Transformers) Average Ranking Error Level 1: 0.76
Model B (HuggingFace Transformers) Average Ranking Error Level 2: 5.20
